# The heavy part of a notebook, on a machine

The notebook stays here; the function that trains runs on a machine of yours
(or in a container on this computer), and its plot comes back in its value.

- `@zenthros.cloud(packages=..., code_dir=None)` goes on the function that trains:
  the packages at the versions this notebook has, and no project files (it
  imports nothing of the project).
- The plot comes back as the image's bytes: `savefig` to an `io.BytesIO`.
- The data here is made on the machine from a seed. With your own data: a small
  DataFrame goes as an argument; a big one as a Parquet with `upload=` (it needs a
  bucket: `zenthros store`). See the skill's `reference/data.md`.

In [ ]:
import importlib.metadata as metadata

import zenthros

# Where it runs: "local" is a container on this computer, with no account;
# or the name of a machine of yours (zenthros targets).
MACHINE = "local"
# The versions this notebook has, pinned on the machine too.
PACKAGES = [f"{name}=={metadata.version(name)}" for name in ("scikit-learn", "matplotlib", "numpy")]
PACKAGES

In [ ]:
@zenthros.cloud(packages=PACKAGES, code_dir=None)
def train(rows=200_000, features=20, seed=0, max_iter=200):
    import io

    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import numpy as np
    from sklearn.datasets import make_regression
    from sklearn.ensemble import HistGradientBoostingRegressor
    from sklearn.model_selection import train_test_split

    x, y = make_regression(n_samples=rows, n_features=features, noise=10.0, random_state=seed)
    x, y = x.astype(np.float32), y.astype(np.float32)
    x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=seed)
    model = HistGradientBoostingRegressor(max_iter=max_iter, random_state=seed).fit(x_train, y_train)
    predicted = model.predict(x_val)
    rmse = float(np.sqrt(np.mean((predicted - y_val) ** 2)))

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(y_val[:2000], predicted[:2000], s=4, alpha=0.5)
    ax.set_xlabel("true")
    ax.set_ylabel("predicted")
    ax.set_title(f"validation, RMSE {rmse:.2f}")
    image = io.BytesIO()
    fig.savefig(image, format="png", dpi=100)
    plt.close(fig)
    return {"rmse": rmse, "plot": image.getvalue()}

In [ ]:
out = train.remote(on=MACHINE)
out["rmse"]

In [ ]:
from IPython.display import Image

Image(out["plot"])

Its job's name was printed as it started (`zenthros: train → zr-train-… on local`):
`zenthros.get(name).result()` reads the value again from another notebook, and
`zenthros logs NAME` its log.